In [1]:
pip install langdetect

Note: you may need to restart the kernel to use updated packages.


In [2]:
from haystack import Pipeline,Document
from haystack.components.classifiers import DocumentLanguageClassifier
from haystack.components.routers import MetadataRouter
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.writers import DocumentWriter


In [3]:
documents =[
    Document(
        content="Super appartement. Juste au dessus de plusieurs bars qui ferment très tard. A savoir à l'avance. (Bouchons d'oreilles fournis !)"
    ),
    Document(
        content="El apartamento estaba genial y muy céntrico, todo a mano. Al lado de la librería Lello y De la Torre de los clérigos. Está situado en una zona de marcha, así que si vais en fin de semana , habrá ruido, aunque a nosotros no nos molestaba para dormir"
    ),
    Document(
        content="The keypad with a code is convenient and the location is convenient. Basically everything else, very noisy, wi-fi didn't work, check-in person didn't explain anything about facilities, shower head was broken, there's no cleaning and everything else one may need is charged."
    ),
    Document(
        content="It is very central and appartement has a nice appearance (even though a lot IKEA stuff), *W A R N I N G** the appartement presents itself as a elegant and as a place to relax, very wrong place to relax - you cannot sleep in this appartement, even the beds are vibrating from the bass of the clubs in the same building - you get ear plugs from the hotel -> now I understand why -> I missed a trip as it was so loud and I could not hear the alarm next day due to the ear plugs.- there is a green light indicating 'emergency exit' just above the bed, which shines very bright at night - during the arrival process, you felt the urge of the agent to leave as soon as possible. - try to go to 'RVA clerigos appartements' -> same price, super quiet, beautiful, city center and very nice staff (not an agency)- you are basically sleeping next to the fridge, which makes a lot of noise, when the compressor is running -> had to switch it off - but then had no cool food and drinks. - the bed was somehow broken down - the wooden part behind the bed was almost falling appart and some hooks were broken before- when the neighbour room is cooking you hear the fan very loud. I initially thought that I somehow activated the kitchen fan"
    ),
    Document(content="Un peu salé surtout le sol. Manque de service et de souplesse"),
    Document(
        content="Nous avons passé un séjour formidable. Merci aux personnes , le bonjours à Ricardo notre taxi man, très sympathique. Je pense refaire un séjour parmi vous, après le confinement, tout était parfait, surtout leur gentillesse, aucune chaude négative. Je n'ai rien à redire de négative, Ils étaient a notre écoute, un gentil message tout les matins, pour nous demander si nous avions besoins de renseignement et savoir si tout allait bien pendant notre séjour."
    ),
    Document(
        content="Céntrico. Muy cómodo para moverse y ver Oporto. Edificio con terraza propia en la última planta. Todo reformado y nuevo. Te traen un estupendo desayuno todas las mañanas al apartamento. Solo que se puede escuchar algo de ruido de la calle a primeras horas de la noche. Es un zona de ocio nocturno. Pero respetan los horarios."
    ),
]

In [4]:
en_document_store = InMemoryDocumentStore()
fr_document_store = InMemoryDocumentStore()
es_document_store = InMemoryDocumentStore()

In [5]:
classifier = DocumentLanguageClassifier(languages = ["en","fr","es"])
route_ruler = {
    "en" : {"field":"meta.language" , "operator":"==" , "value": "en"},
    "fr" : {"field":"meta.language" , "operator":"==" , "value": "fr"},
    "es" : {"field":"meta.language" , "operator":"==" , "value": "es"},
}
router = MetadataRouter(rules=route_ruler)

In [6]:
en_store = DocumentWriter(document_store=en_document_store)
fr_store = DocumentWriter(document_store=fr_document_store)
es_store = DocumentWriter(document_store=es_document_store)

In [7]:
classifier_pipeline = Pipeline()

classifier_pipeline.add_component("classifier",classifier)
classifier_pipeline.add_component("router",router)
classifier_pipeline.add_component("en_store",en_store)
classifier_pipeline.add_component("fr_store",fr_store)
classifier_pipeline.add_component("es_store",es_store)

classifier_pipeline.connect("classifier.documents","router")
classifier_pipeline.connect("router.en","en_store")
classifier_pipeline.connect("router.fr","fr_store")
classifier_pipeline.connect("router.es","es_store")





🚅 Components
  - classifier: DocumentLanguageClassifier
  - router: MetadataRouter
  - en_store: DocumentWriter
  - fr_store: DocumentWriter
  - es_store: DocumentWriter
🛤️ Connections
  - classifier.documents -> router.documents (List[Document])
  - router.en -> en_store.documents (List[Document])
  - router.fr -> fr_store.documents (List[Document])
  - router.es -> es_store.documents (List[Document])

In [8]:
classifier_pipeline.run({"classifier":{"documents":documents}})


{'router': {'unmatched': []},
 'en_store': {'documents_written': 2},
 'fr_store': {'documents_written': 3},
 'es_store': {'documents_written': 2}}

In [9]:
print(fr_document_store.filter_documents())

[Document(id=ea7ea338874232de2d8105a258813f50345db82772e21ad2c4549dbb7adce8a3, content: 'Super appartement. Juste au dessus de plusieurs bars qui ferment très tard. A savoir à l'avance. (Bo...', meta: {'language': 'fr'}), Document(id=6b64c8a60543ee32b81cd39bc8d6e09fae4bff1b22c6ccdcf414db26fa354e7a, content: 'Un peu salé surtout le sol. Manque de service et de souplesse', meta: {'language': 'fr'}), Document(id=b1be23526f19a8af80a190e775bfd05e65878e585529037cb45b47267a4eaa98, content: 'Nous avons passé un séjour formidable. Merci aux personnes , le bonjours à Ricardo notre taxi man, t...', meta: {'language': 'fr'})]


## ----Create RAG Multi-linguil pipeline ------------ ##
 - Detects the language of the query
 - retreive documents related to that lamguage only
 - response to the query based on the retreived documents
   

In [10]:
from haystack.components.routers import TextLanguageRouter
from haystack.components.builders import ChatPromptBuilder
from haystack.components.joiners import DocumentJoiner
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.dataclasses import ChatMessage
from haystack.components.generators.chat import OpenAIChatGenerator


In [11]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
api_key=os.getenv('OPENAI_API_KEY')
llm = OpenAIChatGenerator(model = "gpt-4o-mini")

In [12]:
en_retreiver = InMemoryBM25Retriever(document_store = en_document_store)
fr_retreiver = InMemoryBM25Retriever(document_store = fr_document_store)
es_retreiver = InMemoryBM25Retriever(document_store = es_document_store)

router = TextLanguageRouter(languages = ["en","fr","es"])

joiner = DocumentJoiner()

In [13]:
template = [
    ChatMessage.from_user(
        """
        Given the following information, answer the question.

        Context:
        {% for document in documents %}
            {{ document.content }}
        {% endfor %}
        
        Question: {{question}}
        Answer:
                
        """
    )
]

builder = ChatPromptBuilder(template = template)

In [14]:
pipeline = Pipeline()

pipeline.add_component("router",router)
pipeline.add_component("en_retreiver",en_retreiver)
pipeline.add_component("fr_retreiver",fr_retreiver)
pipeline.add_component("es_retreiver",es_retreiver)
pipeline.add_component("joiner",joiner)
pipeline.add_component("builder",builder)
pipeline.add_component("llm",llm)


In [21]:
pipeline.connect("router.en","en_retreiver")
pipeline.connect("router.fr","fr_retreiver")
pipeline.connect("router.es","es_retreiver")
pipeline.connect("en_retreiver","joiner")
pipeline.connect("fr_retreiver","joiner")
pipeline.connect("es_retreiver","joiner")
pipeline.connect("joiner.documents","builder.documents")
pipeline.connect("builder","llm.messages")


🚅 Components
  - router: TextLanguageRouter
  - en_retreiver: InMemoryBM25Retriever
  - fr_retreiver: InMemoryBM25Retriever
  - es_retreiver: InMemoryBM25Retriever
  - joiner: DocumentJoiner
  - builder: ChatPromptBuilder
  - llm: OpenAIChatGenerator
🛤️ Connections
  - router.en -> en_retreiver.query (str)
  - router.fr -> fr_retreiver.query (str)
  - router.es -> es_retreiver.query (str)
  - en_retreiver.documents -> joiner.documents (List[Document])
  - fr_retreiver.documents -> joiner.documents (List[Document])
  - es_retreiver.documents -> joiner.documents (List[Document])
  - joiner.documents -> builder.documents (List[Document])
  - builder.prompt -> llm.messages (List[ChatMessage])

In [25]:
en_question = "Is this apartment conveniently located?"

en_result = pipeline.run({"router":{"text":en_question},"builder":{"question":en_question}})
print(en_result['llm']['replies'][0].text)

Yes, the apartment is conveniently located. The context describes it as "very central" and mentions that the location is convenient. However, it also highlights significant issues with noise and comfort that may impact the overall experience.
